# Petróleo vs. Bolsa Portuguesa (2000–2026): um Data Lakehouse com Apache Spark

**Unidade curricular:** Analyzing Big Data — EDSA
**Elementos:** Ana Filipa Filipe 20250313, José Justino 20251110, Inês Rodrigues 20251957, Sandra Figueiredo 20251705
**Plataforma:** Apache Spark em Databricks Free Edition (Serverless, Unity Catalog, Delta Lake)

---

### Objetivo
Construir um processo de dados **estruturado e incremental**, como num caso real, para estudar a relação entre o **preço do petróleo** (Brent e WTI) e a **bolsa portuguesa** (índice PSI), de 2000 até hoje.

**Perguntas de análise**
1. Como evoluiu o preço do petróleo desde 2000, em dólares e em euros?
2. O mercado acionista português acompanha o petróleo? Em que períodos a relação é mais forte?
3. Como se comportaram os dois mercados nas grandes crises (2008, 2014-16, COVID-19, guerra na Ucrânia)?
4. Os movimentos do petróleo ajudam a explicar ou a prever os movimentos do PSI?

### Fontes de dados (APIs públicas e gratuitas)

| Fonte | Série | Conteúdo | Frequência | Desde |
|---|---|---|---|---|
| **FRED** (Federal Reserve Bank of St. Louis) | `DCOILBRENTEU` | Brent, preço spot (USD/barril) | Diária | 1987 |
| FRED | `DCOILWTICO` | WTI, preço spot (USD/barril) | Diária | 1986 |
| FRED | `DEXUSEU` | Câmbio USD por 1 EUR | Diária | 1999 |
| FRED / OCDE | `SPASTT01PTM661N` | Índice de preços de ações — Portugal (2015 = 100) | Mensal | 1988 |
| **Yahoo Finance** (chart API) | `PSI20.LS` | Índice PSI (Euronext Lisbon): abertura, máximo, mínimo, fecho, volume | Diária | 2013 |

Nota: não existe uma fonte gratuita com o PSI **diário** antes de 2013 (a Euronext só disponibiliza 2 anos). Por isso o período 2000–2013 é coberto pelo índice mensal da OCDE para Portugal, que acompanha o mercado acionista português.

### Arquitetura (Medallion)

```
 APIs (FRED, Yahoo) ──► LANDING (ficheiros em bruto, 1 por leitura, no Volume)
                             │
                             ▼
                    BRONZE  (Delta, append-only: todo o histórico de leituras + metadados → permite reprocessar)
                             │  normalização: tipos, nulos, duplicados, datas, regras de qualidade
                             ▼
                    SILVER  (Delta, modelo detalhado por tema: dimensão instrumentos + factos diários/mensais, MERGE incremental)
                             │  integração, forward-fill, retornos, médias móveis, agregações
                             ▼
                    GOLD    (Delta, tabelas otimizadas para análise: diária, mensal, anual; particionadas + OPTIMIZE/Z-ORDER)
                             │
                             ▼
                    ANÁLISE (Spark SQL, gráficos/dashboard, modelos Spark MLlib)
```

---
## 0. Configuração do ambiente

**Antes de correr (só na primeira vez):** o notebook tenta ler os dados diretamente das APIs. Como o Databricks Free Edition limita o acesso à internet a domínios "confiáveis", se a API estiver bloqueada o notebook usa **ficheiros de contingência** (*seed*) com a mesma estrutura da resposta da API. Para isso:
1. Correr as células desta secção (criam o schema `workspace.oil_psi` e o Volume `lake`).
2. Em **Catalog → workspace → oil_psi → Volumes → lake**, criar a pasta `seed` e fazer *upload* dos 5 ficheiros da pasta `dados_fonte` (`fred_*.csv` e `yahoo_PSI20.LS.json`).

In [0]:
import os, json, datetime as dt, uuid, math
import requests
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql import types as T

# Deteção do ambiente: no Databricks usamos Delta Lake + Unity Catalog.
# (O notebook também corre em Google Colab / Spark local, guardando as tabelas em Parquet.)
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if not IS_DATABRICKS:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("oil_psi_lakehouse").getOrCreate()
    def display(df, n=20):
        print(df.limit(n).toPandas().to_string())

TABLE_FORMAT = "delta" if IS_DATABRICKS else "parquet"

CATALOG, SCHEMA, VOLUME = "workspace", "oil_psi", "lake"
BASE    = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}" if IS_DATABRICKS else os.path.abspath("lakehouse_oil_psi")
LANDING = f"{BASE}/landing"      # camada de aterragem: ficheiros em bruto exatamente como vieram da API
SEED    = f"{BASE}/seed"         # ficheiros de contingência (snapshot das APIs)

RESET = True    # True = apagar tudo e reconstruir de raiz (demonstração). Em produção ficaria False.

print("Databricks:", IS_DATABRICKS, "| formato das tabelas:", TABLE_FORMAT)
print("Base do lakehouse:", BASE)

Databricks: True | formato das tabelas: delta
Base do lakehouse: /Volumes/workspace/oil_psi/lake


In [0]:
# Criar o schema (base de dados) e o Volume que guarda os ficheiros do lakehouse
if IS_DATABRICKS:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA} COMMENT 'Lakehouse: petróleo vs PSI'")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME} COMMENT 'Ficheiros landing e seed'")
    spark.sql(f"USE CATALOG {CATALOG}")
    spark.sql(f"USE SCHEMA {SCHEMA}")
else:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {SCHEMA}")
    spark.sql(f"USE {SCHEMA}")

ALL_TABLES = ["ctl_ingestion_log", "bz_fred_observations", "bz_yahoo_chart",
              "sv_dim_instrument", "sv_fact_price_daily", "sv_fact_psi_daily", "sv_fact_pt_shares_monthly",
              "gd_market_daily", "gd_market_monthly", "gd_market_yearly"]

if RESET:
    for t in ALL_TABLES:
        spark.sql(f"DROP TABLE IF EXISTS {t}")
    import shutil
    shutil.rmtree(LANDING, ignore_errors=True)   # o seed NÃO é apagado
os.makedirs(LANDING, exist_ok=True)

print("Ficheiros seed disponíveis:", sorted(os.listdir(SEED)) if os.path.isdir(SEED) else "nenhum (só será possível ler das APIs)")

Ficheiros seed disponíveis: nenhum (só será possível ler das APIs)


In [0]:
# Catálogo das séries que alimentam o lakehouse (metadados de negócio)
FRED_SERIES = {
    "DCOILBRENTEU":    {"name": "Brent - preço spot",                         "unit": "USD/barril",   "currency": "USD", "freq": "D", "category": "Petróleo"},
    "DCOILWTICO":      {"name": "WTI - preço spot",                           "unit": "USD/barril",   "currency": "USD", "freq": "D", "category": "Petróleo"},
    "DEXUSEU":         {"name": "Câmbio EUR/USD (USD por 1 EUR)",             "unit": "USD por EUR",  "currency": "USD", "freq": "D", "category": "Câmbio"},
    "SPASTT01PTM661N": {"name": "OCDE - índice de preços de ações Portugal",   "unit": "Índice 2015=100", "currency": "EUR", "freq": "M", "category": "Ações PT"},
}
YAHOO_SYMBOLS = {
    "PSI20.LS":        {"name": "PSI - Euronext Lisbon",                     "unit": "Pontos",       "currency": "EUR", "freq": "D", "category": "Ações PT"},
}

---
## 1. Ingestão: ler os dados da fonte → camada LANDING

Cada leitura (*batch*) chama a API com uma janela de datas `[start, end]` e grava a resposta **sem qualquer alteração** num ficheiro novo:

`landing/<fonte>/<série>/ingest_date=AAAA-MM-DD/<série>_<batch_id>.csv|json`

Assim cada leitura fica guardada para sempre e pode ser reprocessada. Cada leitura também é registada na tabela de controlo `ctl_ingestion_log`.

In [0]:
HTTP_HEADERS = {"User-Agent": "Mozilla/5.0 (EDSA ABD student project)"}

def _http_get(url, params):
    r = requests.get(url, params=params, headers=HTTP_HEADERS, timeout=30)
    r.raise_for_status()
    return r.text

def _to_epoch(d):
    return int(dt.datetime.strptime(d, "%Y-%m-%d").replace(tzinfo=dt.timezone.utc).timestamp())

def fetch_fred(series_id, start, end):
    """Lê uma série do FRED (CSV). Se a API não estiver acessível, usa o ficheiro seed com a mesma janela de datas."""
    try:
        txt = _http_get("https://fred.stlouisfed.org/graph/fredgraph.csv",
                        {"id": series_id, "cosd": start, "coed": end})
        if not txt.startswith("observation_date"):
            raise ValueError("resposta inesperada da API")
        return txt, "API"
    except Exception as e:
        lines = open(find_seed(f"fred_{series_id}.csv"), encoding="utf-8").read().strip().splitlines()
        body = [l for l in lines[1:] if start <= l[:10] <= end]
        return "\n".join([lines[0]] + body) + "\n", f"SEED ({type(e).__name__})"

def fetch_yahoo(symbol, start, end):
    """Lê o histórico diário de um índice do Yahoo Finance (JSON). Contingência: ficheiro seed filtrado pela janela."""
    p1, p2 = _to_epoch(start), _to_epoch(end) + 86400
    try:
        txt = _http_get(f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}",
                        {"period1": p1, "period2": p2, "interval": "1d"})
        json.loads(txt)["chart"]["result"][0]["timestamp"]
        return txt, "API"
    except Exception as e:
        j = json.load(open(find_seed(f"yahoo_{symbol}.json"), encoding="utf-8"))
        res = j["chart"]["result"][0]
        idx = [i for i, t in enumerate(res["timestamp"]) if p1 <= t < p2]
        res["timestamp"] = [res["timestamp"][i] for i in idx]
        for block in ("quote", "adjclose"):
            for serie in res["indicators"].get(block, []):
                for k in serie:
                    serie[k] = [serie[k][i] for i in idx]
        return json.dumps(j), f"SEED ({type(e).__name__})"

def find_seed(filename):
    """Procura o ficheiro seed na pasta seed/ e, se não estiver lá, em qualquer pasta do Volume (ignora maiúsculas)."""
    for root, _, files in os.walk(BASE):
        if "/landing" in root:
            continue
        for f in files:
            if f.lower() == filename.lower():
                return os.path.join(root, f)
    raise FileNotFoundError(f"A API está bloqueada e não encontrei '{filename}' no Volume {BASE}. "
                            f"Faça upload dos ficheiros da pasta dados_fonte para {SEED}/")

def land(source, series, payload, ext, batch_id, ingest_date):
    folder = f"{LANDING}/{source}/{series}/ingest_date={ingest_date}"
    os.makedirs(folder, exist_ok=True)
    path = f"{folder}/{series}_{batch_id}.{ext}"
    with open(path, "w", encoding="utf-8") as f:
        f.write(payload)
    return path

In [0]:
LOG_SCHEMA = T.StructType([
    T.StructField("batch_id", T.StringType()), T.StructField("batch_name", T.StringType()),
    T.StructField("source", T.StringType()), T.StructField("series_id", T.StringType()),
    T.StructField("window_start", T.StringType()), T.StructField("window_end", T.StringType()),
    T.StructField("extract_mode", T.StringType()), T.StructField("records", T.LongType()),
    T.StructField("landing_file", T.StringType()), T.StructField("ingestion_ts", T.TimestampType()),
])

def ingest(batch_name, windows, ingest_date=None):
    """Executa uma leitura (batch) de todas as séries. windows = {series_id: (start, end)}.
    Devolve o batch_id e a lista de ficheiros gravados na landing."""
    batch_id = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None).strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:6]
    ingest_date = ingest_date or dt.date.today().isoformat()
    now = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
    log, files = [], []
    for sid in FRED_SERIES:
        start, end = windows[sid]
        txt, mode = fetch_fred(sid, start, end)
        path = land("fred", sid, txt, "csv", batch_id, ingest_date)
        n = max(len(txt.strip().splitlines()) - 1, 0)
        log.append((batch_id, batch_name, "FRED", sid, start, end, mode, n, path, now)); files.append(("fred", path))
    for sym in YAHOO_SYMBOLS:
        start, end = windows[sym]
        txt, mode = fetch_yahoo(sym, start, end)
        path = land("yahoo", sym, txt, "json", batch_id, ingest_date)
        n = len(json.loads(txt)["chart"]["result"][0].get("timestamp") or [])
        log.append((batch_id, batch_name, "YAHOO", sym, start, end, mode, n, path, now)); files.append(("yahoo", path))
    spark.createDataFrame(log, LOG_SCHEMA).write.format(TABLE_FORMAT).mode("append").saveAsTable("ctl_ingestion_log")
    for r in log:
        print(f"  {r[2]:5} {r[3]:16} {r[4]} → {r[5]}  {r[7]:>6} registos  [{r[6]}]")
    return batch_id, files

### 1.1 Primeira leitura — carga histórica (2000-01-01 a 2025-12-31)
Para simular um processo real, a carga é feita em **três leituras**: a carga histórica inicial agora e duas leituras incrementais na secção 5 (até 30-06-2026 e até hoje).

In [0]:
# Diagnóstico: listar os ficheiros do Volume (exceto a landing) para confirmar que os ficheiros seed estão em seed/
for root, dirs, files in os.walk(BASE):
    if "/landing" not in root:
        print(root, files)

/Volumes/workspace/oil_psi/lake ['fred_DCOILBRENTEU.csv', 'fred_DCOILWTICO.csv', 'fred_DEXUSEU.csv', 'fred_SPASTT01PTM661N.csv', 'yahoo_PSI20.LS.json']


In [0]:
HIST_START, HIST_END = "2000-01-01", "2025-12-31"
all_series = list(FRED_SERIES) + list(YAHOO_SYMBOLS)

batch1_id, batch1_files = ingest("carga_inicial", {s: (HIST_START, HIST_END) for s in all_series})

/home/spark-4b702967-f4c4-484f-94de-70/.ipykernel/62/command-4708452111702422-1963651509:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  batch_id = dt.datetime.utcnow().strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:6]
/home/spark-4b702967-f4c4-484f-94de-70/.ipykernel/62/command-4708452111702422-1963651509:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = dt.datetime.utcnow()


---------------------------------------------------------------------------
gaierror                                  Traceback (most recent call last)
File /databricks/python/lib/python3.12/site-packages/urllib3/connection.py:198, in HTTPConnection._new_conn(self)
    197 try:
--> 198     sock = connection.create_connection(
    199         (self._dns_host, self.port),
    200         self.timeout,
    201         source_address=self.source_address,
    202         socket_options=self.socket_options,
    203     )
    204 except socket.gaierror as e:

File /databricks/python/lib/python3.12/site-packages/urllib3/util/connection.py:60, in create_connection(address, timeout, source_address, socket_options)
     58     raise LocationParseError(f"'{host}', label empty or too long") from None
---> 60 for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
     61     af, socktype, proto, canonname, sa = res

File /usr/lib/python3.12/socket.py:963, in getaddrinfo(host, port, f

In [0]:
# Verificar os ficheiros em bruto na landing (exatamente como vieram da fonte)
for src, path in batch1_files:
    print(path.replace(BASE, "…"), f"{os.path.getsize(path)/1024:.0f} KB")
print()
print(open(batch1_files[0][1]).read()[:200])

---
## 2. Camada BRONZE — histórico de leituras (pronto para reprocessamento)

A Bronze guarda **todas as leituras**, sem descartar nada (*append-only*), mantendo os valores tal como vieram (texto/número em bruto) e acrescentando metadados de linhagem:
`_batch_id`, `_ingestion_ts`, `_source_file`, `_ingest_date`.

- `bz_fred_observations` — uma linha por (série, data, leitura)
- `bz_yahoo_chart` — uma linha por (símbolo, timestamp, leitura), obtida a partir do JSON aninhado da API

In [0]:
def load_bronze_fred(paths, batch_id):
    dfs = []
    for p in paths:
        series_id = p.split("/fred/")[1].split("/")[0]
        raw = spark.read.option("header", True).csv(p)                    # tudo como string (dados em bruto)
        dfs.append(raw.select(
            F.lit(series_id).alias("series_id"),
            F.col("observation_date").alias("observation_date"),
            F.col(series_id).alias("value_raw"),
            F.lit(batch_id).alias("_batch_id"),
            F.current_timestamp().alias("_ingestion_ts"),
            F.col("_metadata.file_path").alias("_source_file"),
            F.regexp_extract(F.col("_metadata.file_path"), r"ingest_date=([0-9-]+)", 1).alias("_ingest_date"),
        ))
    out = dfs[0]
    for d in dfs[1:]:
        out = out.unionByName(d)
    out.write.format(TABLE_FORMAT).mode("append").partitionBy("_ingest_date").saveAsTable("bz_fred_observations")
    return out.count()

def load_bronze_yahoo(paths, batch_id):
    raw = spark.read.option("multiLine", True).json(paths)
    res = raw.select(F.explode("chart.result").alias("r"), F.col("_metadata.file_path").alias("_source_file"))
    q = "r.indicators.quote"
    out = (res.select("*", F.posexplode("r.timestamp").alias("i", "ts_epoch"))
              .select(
                  F.col("r.meta.symbol").alias("symbol"),
                  F.col("r.meta.currency").alias("currency"),
                  F.col("r.meta.exchangeTimezoneName").alias("exchange_tz"),
                  F.col("ts_epoch").cast("long").alias("ts_epoch"),
                  F.col(f"{q}.open")[0][F.col("i")].alias("open_raw"),
                  F.col(f"{q}.high")[0][F.col("i")].alias("high_raw"),
                  F.col(f"{q}.low")[0][F.col("i")].alias("low_raw"),
                  F.col(f"{q}.close")[0][F.col("i")].alias("close_raw"),
                  F.col(f"{q}.volume")[0][F.col("i")].alias("volume_raw"),
                  F.lit(batch_id).alias("_batch_id"),
                  F.current_timestamp().alias("_ingestion_ts"),
                  F.col("_source_file"),
                  F.regexp_extract("_source_file", r"ingest_date=([0-9-]+)", 1).alias("_ingest_date"),
              ))
    out.write.format(TABLE_FORMAT).mode("append").partitionBy("_ingest_date").saveAsTable("bz_yahoo_chart")
    return out.count()

def load_bronze(batch_id, files):
    n1 = load_bronze_fred([p for s, p in files if s == "fred"], batch_id)
    n2 = load_bronze_yahoo([p for s, p in files if s == "yahoo"], batch_id)
    print(f"Bronze ← batch {batch_id}: {n1} linhas FRED, {n2} linhas Yahoo")

load_bronze(batch1_id, batch1_files)

In [0]:
display(spark.table("bz_fred_observations").limit(5))
display(spark.table("bz_yahoo_chart").limit(5))
spark.table("bz_yahoo_chart").printSchema()

### 2.1 Qualidade dos dados em bruto
Antes de normalizar, medimos os problemas que existem na fonte: valores em falta (feriados, dias sem cotação), duplicados e tipos de dados (no FRED tudo chega como texto).

In [0]:
bz_f = spark.table("bz_fred_observations")
display(bz_f.groupBy("series_id").agg(
    F.count("*").alias("linhas"),
    F.sum(F.when((F.col("value_raw").isNull()) | (F.trim("value_raw").isin("", ".")), 1).otherwise(0)).alias("valores_em_falta"),
    F.min("observation_date").alias("primeira_data"), F.max("observation_date").alias("ultima_data"),
    (F.count("*") - F.countDistinct("observation_date")).alias("duplicados"),
).orderBy("series_id"))

bz_y = spark.table("bz_yahoo_chart")
display(bz_y.agg(
    F.count("*").alias("linhas"),
    F.sum(F.when(F.col("close_raw").isNull(), 1).otherwise(0)).alias("fecho_em_falta"),
    F.sum(F.when(F.col("volume_raw") == 0, 1).otherwise(0)).alias("volume_zero"),
    F.from_unixtime(F.min("ts_epoch")).alias("primeiro"), F.from_unixtime(F.max("ts_epoch")).alias("ultimo"),
))

---
## 3. Camada SILVER — dados normalizados e modelados para o tema

**Regras de normalização ("data rules")**
1. Tipos: `value_raw` (texto) → `double`; datas em texto → `date`; *timestamps* Unix → data local de Lisboa.
2. Nulos: valores vazios ou `.` (convenção do FRED para "sem observação") são eliminados; no PSI, dias sem fecho são eliminados.
3. Duplicados: a mesma (série, data) pode chegar em várias leituras. Fica sempre a **leitura mais recente** (`_ingestion_ts`), o que também corrige revisões da fonte.
4. Validação: preços ≤ 0 ficam marcados com `dq_flag` em vez de serem apagados (ex.: o WTI fechou a **–37,63 USD** em 20-04-2020, um valor real).

**Modelo Silver (detalhe máximo, orientado ao tema)**
- `sv_dim_instrument` — dimensão com os instrumentos (série, nome, unidade, moeda, frequência, categoria, fonte)
- `sv_fact_price_daily` — factos diários do petróleo e do câmbio (data, instrumento, valor)
- `sv_fact_psi_daily` — factos diários do PSI (OHLC + volume)
- `sv_fact_pt_shares_monthly` — facto mensal do índice OCDE de ações portuguesas

A escrita é **incremental** com `MERGE` (*upsert*) pelas chaves de negócio.

In [0]:
def upsert(df, table, keys):
    """Upsert incremental: MERGE INTO no Delta Lake (ou substituição equivalente em Parquet fora do Databricks)."""
    if not spark.catalog.tableExists(table):
        df.write.format(TABLE_FORMAT).saveAsTable(table)
        return
    if TABLE_FORMAT == "delta":
        df.createOrReplaceTempView("_src")
        cond = " AND ".join(f"t.{k} = s.{k}" for k in keys)
        spark.sql(f"""MERGE INTO {table} t USING _src s ON {cond}
                      WHEN MATCHED THEN UPDATE SET *
                      WHEN NOT MATCHED THEN INSERT *""")
    else:
        merged = spark.table(table).join(df.select(*keys), keys, "left_anti").unionByName(df).localCheckpoint()
        merged.write.format(TABLE_FORMAT).mode("overwrite").saveAsTable(table)

def latest_per_key(df, keys):
    w = Window.partitionBy(*keys).orderBy(F.col("_ingestion_ts").desc(), F.col("_batch_id").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

In [0]:
# Dimensão de instrumentos (dados de referência)
dim_rows = [(k, v["name"], v["unit"], v["currency"], v["freq"], v["category"], "FRED") for k, v in FRED_SERIES.items()] + \
           [(k, v["name"], v["unit"], v["currency"], v["freq"], v["category"], "Yahoo Finance") for k, v in YAHOO_SYMBOLS.items()]
dim_instrument = spark.createDataFrame(dim_rows, "instrument_id string, name string, unit string, currency string, frequency string, category string, source string")
upsert(dim_instrument, "sv_dim_instrument", ["instrument_id"])
display(spark.table("sv_dim_instrument"))

In [0]:
def normalize_fred(bz):
    df = (bz.withColumn("value", F.when(F.trim("value_raw").isin("", "."), None).otherwise(F.col("value_raw")).cast("double"))
            .withColumn("date", F.to_date("observation_date", "yyyy-MM-dd"))
            .filter(F.col("value").isNotNull() & F.col("date").isNotNull()))
    df = latest_per_key(df, ["series_id", "date"])
    return df.select(
        F.col("series_id").alias("instrument_id"), "date", "value",
        F.when(F.col("value") <= 0, "PRECO_NAO_POSITIVO").alias("dq_flag"),
        "_batch_id", F.col("_ingestion_ts").alias("_source_ingestion_ts"), F.current_timestamp().alias("_updated_ts"))

def normalize_yahoo(bz):
    df = (bz.withColumn("date", F.to_date(F.from_utc_timestamp(F.timestamp_seconds("ts_epoch"), "Europe/Lisbon")))
            .filter(F.col("close_raw").isNotNull()))
    df = latest_per_key(df, ["symbol", "date"])
    return df.select(
        F.col("symbol").alias("instrument_id"), "date",
        F.round(F.col("open_raw").cast("double"), 2).alias("open"),
        F.round(F.col("high_raw").cast("double"), 2).alias("high"),
        F.round(F.col("low_raw").cast("double"), 2).alias("low"),
        F.round(F.col("close_raw").cast("double"), 2).alias("close"),
        F.col("volume_raw").cast("long").alias("volume"),
        F.when(F.col("close_raw") <= 0, "PRECO_NAO_POSITIVO")
         .when(F.col("high_raw") < F.col("low_raw"), "MAX_MENOR_QUE_MIN").alias("dq_flag"),
        "_batch_id", F.col("_ingestion_ts").alias("_source_ingestion_ts"), F.current_timestamp().alias("_updated_ts"))

def build_silver(batch_id=None):
    """Normaliza a Bronze e faz MERGE na Silver. batch_id=None → reprocessa TODO o histórico da Bronze."""
    bz_f = spark.table("bz_fred_observations")
    bz_y = spark.table("bz_yahoo_chart")
    if batch_id:
        bz_f, bz_y = bz_f.filter(F.col("_batch_id") == batch_id), bz_y.filter(F.col("_batch_id") == batch_id)
    fred = normalize_fred(bz_f)
    daily_ids   = [k for k, v in FRED_SERIES.items() if v["freq"] == "D"]
    monthly_ids = [k for k, v in FRED_SERIES.items() if v["freq"] == "M"]
    upsert(fred.filter(F.col("instrument_id").isin(daily_ids)), "sv_fact_price_daily", ["instrument_id", "date"])
    upsert(fred.filter(F.col("instrument_id").isin(monthly_ids)).withColumnRenamed("date", "month_date"),
           "sv_fact_pt_shares_monthly", ["instrument_id", "month_date"])
    upsert(normalize_yahoo(bz_y), "sv_fact_psi_daily", ["instrument_id", "date"])
    for t in ["sv_fact_price_daily", "sv_fact_psi_daily", "sv_fact_pt_shares_monthly"]:
        print(f"{t:28} {spark.table(t).count():>7} linhas")

build_silver(batch1_id)

In [0]:
display(spark.table("sv_fact_price_daily").groupBy("instrument_id").agg(
    F.count("*").alias("dias"), F.min("date").alias("de"), F.max("date").alias("ate"),
    F.round(F.min("value"), 2).alias("min"), F.round(F.max("value"), 2).alias("max"),
    F.sum(F.when(F.col("dq_flag").isNotNull(), 1).otherwise(0)).alias("alertas_qualidade")))
display(spark.table("sv_fact_price_daily").filter("dq_flag IS NOT NULL"))
display(spark.table("sv_fact_psi_daily").orderBy(F.desc("date")).limit(5))

---
## 4. Camada GOLD — dados preparados e otimizados para análise

**Preparação**
- Integração das séries numa **calendário único** de dias úteis (união das datas com cotação).
- *Forward-fill* dos preços em feriados de um dos mercados (ex.: Brent em feriados do Reino Unido, PSI em feriados portugueses), com indicadores `*_quoted` para saber se o valor foi observado.
- **Brent em euros** = Brent USD ÷ EUR/USD (o PSI está em euros, por isso esta é a comparação justa).
- Retornos logarítmicos, médias móveis de 50 e 200 dias, correlação móvel de 1 ano (252 dias úteis), máximos e *drawdowns*.

**Otimização**
- Tabelas largas (*denormalizadas*), sem *joins* na hora da consulta.
- `gd_market_daily` **particionada por ano** + `OPTIMIZE … ZORDER BY (date)` (compactação de ficheiros pequenos e *data skipping*).
- Agregados pré-calculados: `gd_market_monthly` (2000–2026) e `gd_market_yearly`.

In [0]:
def build_gold():
    px = spark.table("sv_fact_price_daily").groupBy("date").pivot("instrument_id", ["DCOILBRENTEU", "DCOILWTICO", "DEXUSEU"]).agg(F.first("value"))
    px = px.withColumnRenamed("DCOILBRENTEU", "brent_usd_obs").withColumnRenamed("DCOILWTICO", "wti_usd_obs").withColumnRenamed("DEXUSEU", "eurusd_obs")
    psi = spark.table("sv_fact_psi_daily").select("date", F.col("close").alias("psi_close_obs"), F.col("volume").alias("psi_volume"))
    spine = px.select("date").union(psi.select("date")).distinct().filter(F.dayofweek("date").between(2, 6))
    d = spine.join(px, "date", "left").join(psi, "date", "left")

    w_ff = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, 0)    # série única e pequena → janela global
    for c in ["brent_usd", "wti_usd", "eurusd", "psi_close"]:
        d = d.withColumn(c, F.last(f"{c}_obs", ignorenulls=True).over(w_ff))
    d = (d.withColumn("brent_quoted", F.col("brent_usd_obs").isNotNull())
          .withColumn("psi_traded", F.col("psi_close_obs").isNotNull())
          .drop("brent_usd_obs", "wti_usd_obs", "eurusd_obs", "psi_close_obs")
          .withColumn("brent_eur", F.round(F.col("brent_usd") / F.col("eurusd"), 4)))

    w = Window.orderBy("date")
    for c, r in [("brent_usd", "ret_brent_usd"), ("brent_eur", "ret_brent_eur"), ("eurusd", "ret_eurusd"), ("psi_close", "ret_psi")]:
        d = d.withColumn(r, F.log(F.col(c) / F.lag(c).over(w)))
    w50, w200, w252 = [Window.orderBy("date").rowsBetween(-(n - 1), 0) for n in (50, 200, 252)]
    w_all = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, 0)
    d = (d.withColumn("brent_eur_ma50", F.avg("brent_eur").over(w50))
          .withColumn("brent_eur_ma200", F.avg("brent_eur").over(w200))
          .withColumn("psi_ma50", F.avg("psi_close").over(w50))
          .withColumn("psi_ma200", F.avg("psi_close").over(w200))
          .withColumn("corr_1y_psi_brent", F.when(F.count("ret_psi").over(w252) >= 200, F.corr("ret_psi", "ret_brent_eur").over(w252)))
          .withColumn("vol_1y_psi", F.stddev("ret_psi").over(w252) * math.sqrt(252))
          .withColumn("vol_1y_brent", F.stddev("ret_brent_eur").over(w252) * math.sqrt(252))
          .withColumn("psi_drawdown", F.col("psi_close") / F.max("psi_close").over(w_all) - 1)
          .withColumn("brent_eur_drawdown", F.col("brent_eur") / F.max("brent_eur").over(w_all) - 1)
          .withColumn("year", F.year("date")).withColumn("quarter", F.quarter("date")).withColumn("month", F.month("date")))
    (d.write.format(TABLE_FORMAT).mode("overwrite").option("overwriteSchema", "true")
       .partitionBy("year").saveAsTable("gd_market_daily"))

    # ---- Mensal (2000+): petróleo + índice OCDE de ações PT + PSI (desde 2013)
    daily = spark.table("gd_market_daily")
    m = (daily.withColumn("month_date", F.trunc("date", "month")).groupBy("month_date").agg(
            F.round(F.avg("brent_usd"), 2).alias("brent_usd_avg"),
            F.round(F.avg("brent_eur"), 2).alias("brent_eur_avg"),
            F.round(F.avg("wti_usd"), 2).alias("wti_usd_avg"),
            F.round(F.avg("eurusd"), 4).alias("eurusd_avg"),
            F.round(F.max_by("brent_eur", "date"), 2).alias("brent_eur_eom"),
            F.round(F.max_by("psi_close", "date"), 2).alias("psi_eom"),
            F.round(F.avg("psi_close"), 2).alias("psi_avg")))
    oecd = spark.table("sv_fact_pt_shares_monthly").select("month_date", F.round("value", 2).alias("pt_shares_idx"))
    wm = Window.orderBy("month_date")
    m = (m.join(oecd, "month_date", "left")
          .withColumn("ret_brent_eur_m", F.col("brent_eur_avg") / F.lag("brent_eur_avg").over(wm) - 1)
          .withColumn("ret_pt_shares_m", F.col("pt_shares_idx") / F.lag("pt_shares_idx").over(wm) - 1)
          .withColumn("ret_psi_m", F.col("psi_eom") / F.lag("psi_eom").over(wm) - 1)
          .withColumn("yoy_brent_eur", F.col("brent_eur_avg") / F.lag("brent_eur_avg", 12).over(wm) - 1)
          .withColumn("yoy_pt_shares", F.col("pt_shares_idx") / F.lag("pt_shares_idx", 12).over(wm) - 1)
          .withColumn("year", F.year("month_date")))
    m.write.format(TABLE_FORMAT).mode("overwrite").option("overwriteSchema", "true").saveAsTable("gd_market_monthly")

    # ---- Anual
    # variação anual = valor no fim do ano / valor no fim do ano anterior - 1
    wy = Window.orderBy("year")
    y = (daily.groupBy("year").agg(
            F.round(F.avg("brent_usd"), 2).alias("brent_usd_avg"),
            F.round(F.min("brent_usd"), 2).alias("brent_usd_min"),
            F.round(F.max("brent_usd"), 2).alias("brent_usd_max"),
            F.round(F.avg("brent_eur"), 2).alias("brent_eur_avg"),
            F.round(F.avg("eurusd"), 4).alias("eurusd_avg"),
            F.max_by("brent_eur", "date").alias("brent_eur_eoy"),
            F.max_by("psi_close", "date").alias("psi_eoy"),
            F.round(F.stddev("ret_brent_eur") * math.sqrt(252), 4).alias("vol_brent_eur"),
            F.round(F.stddev("ret_psi") * math.sqrt(252), 4).alias("vol_psi"),
            F.round(F.corr("ret_psi", "ret_brent_eur"), 3).alias("corr_diaria_psi_brent"),
            F.sum(F.col("psi_traded").cast("int")).alias("dias_psi"))
         .join(spark.table("sv_fact_pt_shares_monthly").groupBy(F.year("month_date").alias("year"))
                    .agg(F.max_by("value", "month_date").alias("pt_shares_eoy")), "year", "left")
         .withColumn("var_brent_eur", F.round(F.col("brent_eur_eoy") / F.lag("brent_eur_eoy").over(wy) - 1, 4))
         .withColumn("var_psi", F.round(F.col("psi_eoy") / F.lag("psi_eoy").over(wy) - 1, 4))
         .withColumn("var_pt_shares_oecd", F.round(F.col("pt_shares_eoy") / F.lag("pt_shares_eoy").over(wy) - 1, 4)))
    y.write.format(TABLE_FORMAT).mode("overwrite").option("overwriteSchema", "true").saveAsTable("gd_market_yearly")

    if TABLE_FORMAT == "delta":
        spark.sql("OPTIMIZE gd_market_daily ZORDER BY (date)")
    for t in ["gd_market_daily", "gd_market_monthly", "gd_market_yearly"]:
        print(f"{t:20} {spark.table(t).count():>6} linhas")

build_gold()

In [0]:
display(spark.table("gd_market_daily").orderBy(F.desc("date")).limit(5))

---
## 5. Leituras incrementais e reprocessamento

Num processo real o pipeline corre periodicamente. Aqui fazemos mais **duas leituras** da fonte, cada uma a partir da última data já existente na Bronze (*watermark*) com uma **sobreposição de 45 dias**. A sobreposição garante que correções da fonte são apanhadas, mas cria duplicados na Bronze, que a Silver resolve com o `MERGE`.

In [0]:
OVERLAP_DAYS = 45

def watermarks():
    wf = spark.table("bz_fred_observations").groupBy("series_id").agg(F.max("observation_date").alias("wm"))
    wy = spark.table("bz_yahoo_chart").groupBy("symbol").agg(F.to_date(F.from_unixtime(F.max("ts_epoch"))).cast("string").alias("wm"))
    return {r[0]: r[1] for r in wf.collect() + wy.collect()}

def run_pipeline(batch_name, end_date):
    """Pipeline completo de uma leitura incremental: API → Landing → Bronze → Silver (MERGE) → Gold."""
    wm = watermarks()
    windows = {}
    for s in all_series:
        start = (dt.date.fromisoformat(wm[s]) - dt.timedelta(days=OVERLAP_DAYS)).isoformat() if s in wm else HIST_START
        windows[s] = (start, end_date)
    print(f"=== {batch_name}: leitura até {end_date}")
    bid, files = ingest(batch_name, windows)
    load_bronze(bid, files)
    build_silver(bid)
    build_gold()
    return bid

batch2_id = run_pipeline("incremental_1", "2026-06-30")

In [0]:
batch3_id = run_pipeline("incremental_2", dt.date.today().isoformat())

In [0]:
# Registo de todas as leituras (tabela de controlo do processo)
display(spark.table("ctl_ingestion_log").select("batch_name", "source", "series_id", "window_start", "window_end", "extract_mode", "records", "ingestion_ts")
        .orderBy("ingestion_ts", "series_id"))

In [0]:
# A Bronze guarda TODAS as leituras (com sobreposições); a Silver tem um único registo por chave
bz_rows = spark.table("bz_fred_observations").groupBy("series_id").agg(
    F.countDistinct("_batch_id").alias("leituras"), F.count("*").alias("linhas_bronze"),
    F.countDistinct("observation_date").alias("datas_distintas"))
sv_rows = (spark.table("sv_fact_price_daily").select("instrument_id").unionAll(spark.table("sv_fact_pt_shares_monthly").select("instrument_id"))
           .groupBy(F.col("instrument_id").alias("series_id")).agg(F.count("*").alias("linhas_silver")))
display(bz_rows.join(sv_rows, "series_id")
        .withColumn("duplicados_removidos", F.col("linhas_bronze") - F.col("datas_distintas"))
        .withColumn("sem_valor_removidos", F.col("datas_distintas") - F.col("linhas_silver")))

### 5.1 Reprocessamento a partir da Bronze
Como a Bronze tem todo o histórico de leituras, a Silver pode ser **reconstruída de raiz** a qualquer momento (por exemplo, se uma regra de normalização mudar). O resultado tem de ser idêntico ao obtido incrementalmente.

In [0]:
before = {t: spark.table(t).count() for t in ["sv_fact_price_daily", "sv_fact_psi_daily", "sv_fact_pt_shares_monthly"]}
for t in before:
    spark.sql(f"DROP TABLE IF EXISTS {t}")
build_silver(batch_id=None)          # None = todo o histórico da Bronze
after = {t: spark.table(t).count() for t in before}
print("Silver idêntica após reprocessamento:", before == after)

In [0]:
# Delta Lake: histórico de versões da tabela (time travel) — só no Databricks
if TABLE_FORMAT == "delta":
    display(spark.sql("DESCRIBE HISTORY gd_market_daily").select("version", "timestamp", "operation", "operationMetrics"))
    print("Linhas na versão 0 (após a carga inicial):", spark.sql("SELECT COUNT(*) FROM gd_market_daily VERSION AS OF 0").first()[0])
    print("Linhas na versão atual:", spark.table("gd_market_daily").count())

---
## 6. Análise
### 6.1 Queries (Spark SQL sobre a camada Gold)

**Resumo anual:** preço médio do Brent em USD e em EUR, variação anual do Brent (em EUR), do PSI e do índice OCDE de ações PT, volatilidade e correlação diária PSI–Brent.

In [0]:
%sql
SELECT year, brent_usd_avg, brent_usd_min, brent_usd_max, brent_eur_avg, eurusd_avg,
       ROUND(var_brent_eur*100,1) AS var_brent_eur_pct,
       ROUND(var_pt_shares_oecd*100,1) AS var_acoes_pt_oecd_pct,
       ROUND(var_psi*100,1) AS var_psi_pct,
       ROUND(vol_psi*100,1) AS vol_psi_pct, corr_diaria_psi_brent
FROM gd_market_yearly
ORDER BY year

**Comportamento nas grandes crises:** variação do Brent (EUR) e das ações portuguesas entre o início e o fim de cada episódio (dados mensais, para cobrir 2000–2026).

In [0]:
%sql
WITH episodios AS (
  SELECT * FROM VALUES
    ('Bolha dot-com',              DATE'2000-03-01', DATE'2003-03-01'),
    ('Subida do petróleo 2007-08', DATE'2007-01-01', DATE'2008-07-01'),
    ('Crise financeira 2008',      DATE'2008-07-01', DATE'2009-03-01'),
    ('Crise da dívida (troika)',   DATE'2010-01-01', DATE'2012-06-01'),
    ('Colapso do petróleo 2014-16',DATE'2014-06-01', DATE'2016-01-01'),
    ('COVID-19',                   DATE'2020-01-01', DATE'2020-04-01'),
    ('Recuperação pós-COVID',      DATE'2020-04-01', DATE'2021-12-01'),
    ('Guerra na Ucrânia',          DATE'2022-01-01', DATE'2022-06-01'),
    ('2026 até hoje',              DATE'2026-01-01', DATE'2026-12-01')
  AS t(episodio, inicio, fim))
SELECT e.episodio, e.inicio, MAX(m.month_date) AS fim_dados,
       ROUND((MAX_BY(m.brent_eur_avg, m.month_date)/MIN_BY(m.brent_eur_avg, m.month_date)-1)*100,1) AS var_brent_eur_pct,
       ROUND((MAX_BY(m.pt_shares_idx, m.month_date) FILTER (WHERE m.pt_shares_idx IS NOT NULL)
             /MIN_BY(m.pt_shares_idx, m.month_date) FILTER (WHERE m.pt_shares_idx IS NOT NULL)-1)*100,1) AS var_acoes_pt_pct,
       ROUND((MAX_BY(m.psi_avg, m.month_date) FILTER (WHERE m.psi_avg IS NOT NULL)
             /MIN_BY(m.psi_avg, m.month_date) FILTER (WHERE m.psi_avg IS NOT NULL)-1)*100,1) AS var_psi_pct
FROM episodios e JOIN gd_market_monthly m ON m.month_date BETWEEN e.inicio AND e.fim
GROUP BY e.episodio, e.inicio ORDER BY e.inicio

**Os 10 dias com maior variação do Brent** (em EUR) desde 2013 e o que o PSI fez nesse mesmo dia.

In [0]:
%sql
SELECT date, brent_usd, ROUND(ret_brent_eur*100,2) AS brent_eur_pct, psi_close, ROUND(ret_psi*100,2) AS psi_pct
FROM gd_market_daily
WHERE year >= 2013 AND brent_quoted AND psi_traded
ORDER BY ABS(ret_brent_eur) DESC
LIMIT 10

**Regimes do petróleo:** retorno médio mensal das ações portuguesas quando o Brent (EUR) sobe muito, fica estável ou cai muito nesse mês.

In [0]:
%sql
SELECT CASE WHEN ret_brent_eur_m <= -0.10 THEN '1. Brent cai > 10%'
            WHEN ret_brent_eur_m <  -0.03 THEN '2. Brent cai 3-10%'
            WHEN ret_brent_eur_m <=  0.03 THEN '3. Brent estável (±3%)'
            WHEN ret_brent_eur_m <   0.10 THEN '4. Brent sobe 3-10%'
            ELSE '5. Brent sobe > 10%' END AS regime_petroleo,
       COUNT(*) AS meses,
       ROUND(AVG(ret_pt_shares_m)*100,2) AS ret_medio_acoes_pt_pct,
       ROUND(AVG(CASE WHEN ret_pt_shares_m > 0 THEN 1 ELSE 0 END)*100,0) AS pct_meses_acoes_sobem
FROM gd_market_monthly
WHERE ret_brent_eur_m IS NOT NULL AND ret_pt_shares_m IS NOT NULL
GROUP BY 1 ORDER BY 1

**Correlação** entre retornos do petróleo e das ações portuguesas, por frequência e período.

In [0]:
dly = spark.table("gd_market_daily").filter("year >= 2013 AND psi_traded")
mth = spark.table("gd_market_monthly")
corr_df = (dly.agg(F.corr("ret_psi", "ret_brent_eur").alias("c")).select(F.lit("Diária 2013–2026: PSI vs Brent EUR").alias("medida"), "c")
    .union(dly.agg(F.corr("ret_psi", "ret_eurusd")).select(F.lit("Diária 2013–2026: PSI vs EUR/USD"), "*"))
    .union(mth.agg(F.corr("ret_pt_shares_m", "ret_brent_eur_m")).select(F.lit("Mensal 2000–2026: ações PT (OCDE) vs Brent EUR"), "*"))
    .union(mth.agg(F.corr("ret_psi_m", "ret_brent_eur_m")).select(F.lit("Mensal 2013–2026: PSI vs Brent EUR"), "*"))
    .union(mth.agg(F.corr("pt_shares_idx", "brent_eur_avg")).select(F.lit("Níveis mensais 2000–2026: ações PT vs Brent EUR"), "*"))
    .withColumn("correlacao", F.round("c", 3)).drop("c"))
display(corr_df)

### 6.2 Dashboard
Os gráficos abaixo são gerados a partir das tabelas Gold (ficam no HTML exportado).

No Databricks foi também criado um **AI/BI Dashboard** (menu *New → Dashboard*) com estas fontes de dados: `gd_market_daily`, `gd_market_monthly` e `gd_market_yearly`. Tem 4 widgets:
1. Brent (USD e EUR) mensal 2000–2026 (linhas)
2. Índice OCDE ações PT vs Brent EUR (linhas, eixo duplo)
3. Correlação móvel de 1 ano PSI–Brent (linha)
4. Tabela de resumo anual (`gd_market_yearly`)

In [0]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pm = spark.table("gd_market_monthly").orderBy("month_date").toPandas()
pdl = spark.table("gd_market_daily").filter("year >= 2013").orderBy("date").toPandas()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

ax = axes[0, 0]
ax.plot(pm.month_date, pm.brent_usd_avg, label="Brent (USD/barril)", color="#1f4e79")
ax.plot(pm.month_date, pm.brent_eur_avg, label="Brent (EUR/barril)", color="#e07b00")
ax.set_title("Preço do Brent, média mensal 2000–2026"); ax.legend(); ax.grid(alpha=.3)

ax = axes[0, 1]
ax.plot(pm.month_date, pm.pt_shares_idx, color="#2e7d32", label="Ações PT – OCDE (2015=100)")
ax.set_ylabel("Índice ações PT"); ax.grid(alpha=.3)
ax2 = ax.twinx(); ax2.plot(pm.month_date, pm.brent_eur_avg, color="#e07b00", alpha=.6, label="Brent (EUR)")
ax2.set_ylabel("Brent EUR/barril")
ax.set_title("Ações portuguesas vs Brent em EUR (mensal)")
ax.legend(loc="upper left"); ax2.legend(loc="upper right")

ax = axes[1, 0]
base = pdl.dropna(subset=["psi_close"]).iloc[0]
ax.plot(pdl.date, pdl.psi_close / base.psi_close * 100, label="PSI", color="#2e7d32")
ax.plot(pdl.date, pdl.brent_eur / base.brent_eur * 100, label="Brent (EUR)", color="#e07b00")
ax.set_title(f"PSI vs Brent EUR, base 100 = {base.date}"); ax.legend(); ax.grid(alpha=.3)

ax = axes[1, 1]
ax.plot(pdl.date, pdl.corr_1y_psi_brent, color="#6a1b9a")
ax.axhline(0, color="grey", lw=.8)
ax.set_title("Correlação móvel de 1 ano (252 dias) dos retornos diários PSI vs Brent EUR"); ax.grid(alpha=.3)

plt.tight_layout(); plt.show()

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
ax = axes[0]
mm = pm.dropna(subset=["ret_brent_eur_m", "ret_pt_shares_m"])
ax.scatter(mm.ret_brent_eur_m * 100, mm.ret_pt_shares_m * 100, s=12, alpha=.6, color="#1f4e79")
ax.axhline(0, color="grey", lw=.8); ax.axvline(0, color="grey", lw=.8)
ax.set_xlabel("Retorno mensal Brent EUR (%)"); ax.set_ylabel("Retorno mensal ações PT (%)")
ax.set_title("Retornos mensais 2000–2026"); ax.grid(alpha=.3)

ax = axes[1]
py = spark.table("gd_market_yearly").orderBy("year").toPandas()
x = py.year.astype(int)
ax.bar(x - 0.2, py.var_brent_eur * 100, width=.4, label="Brent EUR", color="#e07b00")
ax.bar(x + 0.2, py.var_pt_shares_oecd * 100, width=.4, label="Ações PT (OCDE)", color="#2e7d32")
ax.axhline(0, color="grey", lw=.8)
ax.set_title("Variação anual (%)"); ax.legend(); ax.grid(alpha=.3, axis="y")
plt.tight_layout(); plt.show()

### 6.3 Modelos de Machine Learning (Spark MLlib)

**Modelo 1: regressão linear (explicativo).** Quanto do movimento diário do PSI é explicado, no mesmo dia, pelo petróleo (Brent EUR) e pelo câmbio EUR/USD? Período 2013–2026.

**Modelo 2: Random Forest (preditivo).** Com a informação disponível até hoje (retornos passados do PSI e do Brent, distância às médias móveis, volatilidade), conseguimos prever se o PSI **sobe amanhã**? Divisão temporal: treino 2013–2023 e teste 2024–2026, sem misturar futuro com passado.

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

ml_base = spark.table("gd_market_daily").filter("year >= 2013 AND psi_traded AND ret_psi IS NOT NULL AND ret_brent_eur IS NOT NULL AND ret_eurusd IS NOT NULL")

lr_df = VectorAssembler(inputCols=["ret_brent_eur", "ret_eurusd"], outputCol="features").transform(ml_base)
lr_model = LinearRegression(featuresCol="features", labelCol="ret_psi").fit(lr_df)
print(f"Regressão linear  ret_psi = {lr_model.intercept:.5f} + {lr_model.coefficients[0]:.4f}·ret_brent_eur + {lr_model.coefficients[1]:.4f}·ret_eurusd")
print(f"R² = {lr_model.summary.r2:.3f}  |  RMSE = {lr_model.summary.rootMeanSquaredError:.4f}  |  n = {lr_model.summary.numInstances}")
print(f"Leitura: quando o Brent (EUR) sobe 10%, o PSI sobe em média {lr_model.coefficients[0]*10:.2f}% no mesmo dia.")

In [0]:
w = Window.orderBy("date")
feat = ml_base
# psi_lag1 = retorno de hoje (conhecido no fecho), psi_lag2 = ontem, ... ; o alvo é o retorno de amanhã
feat = feat.withColumn("psi_lag1", F.col("ret_psi")).withColumn("brent_lag1", F.col("ret_brent_eur"))
for k in range(2, 6):
    feat = feat.withColumn(f"psi_lag{k}", F.lag("ret_psi", k - 1).over(w)).withColumn(f"brent_lag{k}", F.lag("ret_brent_eur", k - 1).over(w))
feat = (feat.withColumn("psi_vs_ma50", F.col("psi_close") / F.col("psi_ma50") - 1)
            .withColumn("psi_vs_ma200", F.col("psi_close") / F.col("psi_ma200") - 1)
            .withColumn("brent_vs_ma50", F.col("brent_eur") / F.col("brent_eur_ma50") - 1)
            .withColumn("label", (F.lead("ret_psi", 1).over(w) > 0).cast("double"))
            .dropna())
features = [f"psi_lag{k}" for k in range(1, 6)] + [f"brent_lag{k}" for k in range(1, 6)] + \
           ["ret_eurusd", "psi_vs_ma50", "psi_vs_ma200", "brent_vs_ma50", "vol_1y_psi", "vol_1y_brent", "corr_1y_psi_brent"]
data = VectorAssembler(inputCols=features, outputCol="features").transform(feat)
train, test = data.filter("year <= 2023"), data.filter("year >= 2024")

rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=200, maxDepth=5, seed=42).fit(train)
pred = rf.transform(test)
auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC").evaluate(pred)
acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy").evaluate(pred)
baseline = test.agg(F.avg("label")).first()[0]
print(f"Treino: {train.count()} dias | Teste: {test.count()} dias")
print(f"Accuracy = {acc:.3f} | AUC = {auc:.3f} | baseline 'sobe sempre' = {max(baseline, 1-baseline):.3f}")

imp = sorted(zip(features, rf.featureImportances.toArray()), key=lambda x: -x[1])[:8]
display(spark.createDataFrame([(f, float(round(v, 3))) for f, v in imp], "variavel string, importancia double"))

---
## 7. Conclusões

_Os valores citados são da execução de 07-11-2026. Se o notebook voltar a correr mais tarde, os últimos meses podem mudar ligeiramente._

### 7.1 Sobre os dados

**1. Petróleo: 26 anos de ciclos muito violentos.** O Brent passou de uma média de **28,6 USD** em 2000 para máximos de **144 USD** em julho de 2008. A seguir vieram as quedas para **34 USD** no fim de 2008, para **26 USD** em 2016 (colapso de 2014-16) e para um mínimo de **9 USD** na COVID-19 (abril de 2020). Teve nova subida em 2022 (média de **101 USD**) e voltou a subir em 2026 (média anual de **97 USD** até agora e máximo de **138 USD**; as causas de 2026 ficam fora do âmbito destes dados).

**2. O câmbio importa.** Para um investidor europeu, a valorização do euro entre 2002 e 2008 amorteceu a subida: em 2008 a média foi de 97 USD mas só **65 EUR** por barril. Comparar o petróleo com o PSI em euros é a forma correta. Na prática, o EUR/USD quase não se correlaciona com o PSI diário (0,04).

**3. A relação petróleo–bolsa portuguesa existe, mas é fraca e instável.**
- A correlação dos retornos é de **0,21** (diária, 2013–2026) e de **0,25** (mensal, 2000–2026).
- A correlação dos níveis é mais alta (0,51), mas reflete sobretudo tendências comuns de longo prazo, não causalidade.
- A correlação móvel de 1 ano foi positiva entre 2014 e 2023, com pico perto de **0,55 em 2016** (colapso do petróleo, com peso relevante do setor energético no PSI). Depois desceu para zero e ficou **ligeiramente negativa em 2024–2026**: em 2026 o Brent em euros subiu **83%** e o PSI subiu **15%**.

**4. O petróleo funciona como termómetro do ciclo económico.** Nos meses em que o Brent cai mais de 10%, as ações portuguesas recuam em média **–1,6%** e só sobem em 42% dos meses. Nos meses em que o Brent sobe mais de 10%, avançam **+2,4%** e sobem em 72% dos meses. Petróleo e ações tendem a cair juntos nas crises de procura (2008: Brent –60%, ações PT –51%; COVID-19: –70% e –17%) e a recuperar juntos (2020–21).

**5. Há exceções importantes quando o choque é de oferta ou é interno.** Na subida do petróleo de 2007-08 (Brent +104%) as ações PT caíram 20%. Durante a crise da dívida e a troika (2010–12) o Brent subiu 42% e as ações caíram 34%. Nestes episódios o petróleo caro é um custo para a economia, ou os fatores domésticos dominam.

**6. Prever o PSI com o petróleo não funciona.**
- A regressão linear mostra que o Brent e o câmbio explicam só **5% (R²)** da variação diária do PSI: uma subida de 10% do Brent associa-se a +0,8% do PSI no mesmo dia.
- O Random Forest para prever se o PSI sobe no dia seguinte teve **accuracy de 47,2% e AUC de 0,45**. Ficou abaixo da regra ingénua "sobe sempre" (53,4%).
- É o resultado esperado num mercado eficiente: a informação pública do passado não dá vantagem para prever o dia seguinte.

### 7.2 Sobre o processo
- **Medallion de ponta a ponta:** Landing (ficheiros em bruto) → Bronze (histórico *append-only*) → Silver (modelo normalizado) → Gold (tabelas analíticas).
- **Incremental:** foram feitas 3 leituras da fonte (carga histórica + 2 incrementais), cada uma a partir do *watermark* com 45 dias de sobreposição.
  - A Bronze acumulou as sobreposições: 65 duplicados por série diária.
  - O `MERGE` da Silver resolveu-os, ficando a versão mais recente de cada registo.
- **Reprocessável:** a Silver foi apagada e reconstruída só a partir da Bronze, com resultado idêntico. É isto que permite corrigir regras de negócio sem voltar à fonte.
- **Qualidade e rastreabilidade:**
  - Foram removidos centenas de dias sem cotação (feriados) e convertidos tipos de texto para número e data.
  - Valores anómalos mas reais (WTI a –37,63 USD em 20-04-2020) ficaram marcados com `dq_flag` em vez de apagados.
  - Cada registo guarda o `_batch_id` e o ficheiro de origem, e a tabela `ctl_ingestion_log` documenta todas as leituras.
- **Limitação de dados:** não há PSI diário gratuito antes de 2013. A solução foi combinar duas fontes: o índice mensal da OCDE para 2000–2026 e o PSI diário para 2013–2026.

### 7.3 Sobre a tecnologia
- **Apache Spark** permitiu fazer tudo com a mesma API, em Python e SQL:
  - ler CSV e JSON aninhado (`explode`/`posexplode`);
  - funções de janela (forward-fill, retornos, médias móveis, correlação móvel);
  - modelos de ML (MLlib).
  O volume aqui é pequeno (~7 mil dias), mas o mesmo código escala para dados intradiários ou para milhares de ativos sem alterações.
- **Delta Lake** trouxe garantias de base de dados aos ficheiros: transações ACID, `MERGE` para *upserts*, *time travel* (`DESCRIBE HISTORY`, `VERSION AS OF`) e `OPTIMIZE`/`ZORDER` para otimizar a leitura da Gold.
- **Databricks Free Edition** chegou para o projeto (Unity Catalog, Volumes, SQL, dashboards), mas tem limitações:
  - o acesso à internet está restrito, por isso foi criado um mecanismo de contingência com ficheiros *seed* que mantém o mesmo formato da API;
  - o compute *serverless* não permite `cache()`;
  - a escala de compute é reduzida.
- **Melhorias futuras:**
  - agendar o pipeline como *Job* diário;
  - usar *Auto Loader* / *Lakeflow Declarative Pipelines* para a ingestão;
  - acrescentar mais variáveis explicativas (taxas de juro, PSI por setor, Galp vs. restantes cotadas);
  - modelar a volatilidade em vez da direção do mercado.